# dfthcsos: (8, 143, C) with more copies (commit mode)

Run it with **Save Version → Save & Run All (Commit)**. Everything written to `/kaggle/working` stays in the
version's **Output** tab, and after every step the best verified result so far is packaged as
`/kaggle/working/submission.zip`.

Inputs (right panel, **+ Add Input**): the competition, the *Verifiable Circuit Compilation Starter Kit* dataset and
the `dfthcsos-toolkit` dataset (toolkit v5, unchanged). Settings: **GPU** accelerator, **Internet** on.

Why: in v3, (8, 143, 44) got stuck at 35 mHa. Copies are cheap in the cost model (at lambda_eff ~ 59 and 12 bits,
C = 44 / 80 / 100 score about x2.33 / x2.20 / x2.14), and in a local test C = 100 reached a ~30% lower error than
C = 44 after the same number of steps.

Plan, for C = 100 and then C = 80:
1. Stage 1 (100k epochs) from an eigendecomposition-based start built here with `df_init.py`.
2. Stage 2 (200k epochs) to refine the error.
3. Weight refit (QAT) at 12 bits of the stage 2 result, only if it did not pass.


In [ ]:
# Settings. EPOCH_SCALE < 1 shortens every run (only for quick tests); keep it at 1.0 for real runs.
EPOCH_SCALE = 1.0

In [ ]:
# 1. Locate the inputs and prepare the work folders
import functools, json, os, shutil, subprocess, sys, traceback, zipfile
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

def find(pattern, what):
    hits = list(INPUT.glob(pattern))
    if not hits:
        raise FileNotFoundError(f'{what} not found: add it with "+ Add Input" (right panel).')
    return hits[0]

OFFICIAL = INPUT / 'competitions' / 'google-verifiable-quantum-circuit-compilation'
COMPETITION = OFFICIAL if (OFFICIAL / 'verifiers').is_dir() else \
    find('**/verifiers/dfthcsos/verify.py', 'the competition data').parents[2]
KIT_SRC = find('**/starter_kit/starter', 'the starter kit').parent
TOOLKIT_SRC = find('**/init_8_143_44*', 'the dfthcsos-toolkit dataset (v5, with init_8_143_44.npz)').parent
print('competition :', COMPETITION)
print('starter kit :', KIT_SRC)
print('toolkit     :', TOOLKIT_SRC)

# The toolkit is copied to a writable folder.
TOOLKIT = WORK / 'toolkit'
shutil.copytree(TOOLKIT_SRC, TOOLKIT, dirs_exist_ok=True)

# The trainer looks for the FCIDUMP and the fixed shifts in the folder above starter_kit/.
KIT_PARENT = WORK / 'kit'; KIT_PARENT.mkdir(exist_ok=True)
for link, target in [('starter_kit', KIT_SRC),
                     ('FeMoCo76o113e.fcidump', COMPETITION / 'verifiers/dfthcsos/FeMoCo76o113e.fcidump'),
                     ('FeMoCo76o113e.fixed_shifts.npz', COMPETITION / 'verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz')]:
    if not (KIT_PARENT / link).exists():
        (KIT_PARENT / link).symlink_to(target)

os.environ.update({'COMPETITION_DIR': str(COMPETITION),
                   'KIT_DIR': str(KIT_PARENT / 'starter_kit'),
                   'WORK_DIR': str(WORK)})
print('ok')

In [ ]:
# 2. Check the GPU and the libraries
import importlib.util
if importlib.util.find_spec('optax') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optax'], check=True)
import jax
print('JAX backend:', jax.default_backend())  # should be 'gpu'

In [ ]:
# 3. Load the toolkit and define the helpers
import numpy as np

sys.path.insert(0, str(TOOLKIT))
os.chdir(TOOLKIT)
import run_experiments as RX
import starter.task1_factorization.train_df_thc_qaggle as TQ
from dfthc import train_df_thc_sga as TS
from qat import qat as _qat

REF = RX.lab.ref()
SORTED_SEED = TOOLKIT / 'FeMoCo76o113e_65_60_60_sorted.npz'
FACTORIZATION_KEYS = ('U', 'W', 'H2_I', 'H1_I', 'Hsym_lower')


def log(record):
    """Print a result and append it to results.jsonl."""
    print(json.dumps(record, indent=1, default=float))
    with open(WORK / 'results.jsonl', 'a') as f:
        f.write(json.dumps(record, default=float) + '\n')


def package():
    """Write submission.zip from the best verified result so far (reference circuits for invsqrt and dqi)."""
    results = WORK / 'results.jsonl'
    rows = [json.loads(line) for line in open(results)] if results.exists() else []
    for r in rows:
        print(f"{r['name']:14s} status={str(r.get('status')):20s} brot={r.get('brot')} "
              f"improvement={r.get('improvement', float('nan')):.4f}")
    passed = [r for r in rows if r.get('status') == 'PASS']
    if not passed:
        print('No verified result yet: submission.zip not written.')
        return
    best = max(passed, key=lambda r: r['improvement'])
    sample = COMPETITION / 'sample_submission'
    with zipfile.ZipFile(WORK / 'submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for name in ('invsqrt.kmb', 'dqi.kmb'):
            zf.write(sample / name, name)
        for path in sorted(Path(best['folder']).iterdir()):
            zf.write(path, f'dfthcsos/{path.name}')
    print(f"submission.zip <- {best['name']} (improvement x{best['improvement']:.4f})")


def seed_npz(name, search_root=INPUT, out_dir=WORK / 'seeds'):
    """Return a .npz for factorization `name`, rebuilding it if Kaggle unpacked it into a folder of .npy files."""
    files = [p for p in Path(search_root).glob(f'**/{name}.npz') if p.is_file()]
    if files:
        arrays = dict(np.load(files[0]))
    else:
        folders = [p for p in Path(search_root).glob(f'**/{name}') if (p / 'U.npy').is_file()]
        if not folders:
            raise FileNotFoundError(f'{name}: neither {name}.npz nor a folder with U.npy under {search_root}')
        arrays = {key: np.load(folders[0] / f'{key}.npy') for key in FACTORIZATION_KEYS}
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    path = Path(out_dir) / f'{name}.npz'
    np.savez(path, **{key: arrays[key] for key in FACTORIZATION_KEYS})
    print('seed:', path, arrays['U'].shape)
    return path


def build_only(name, source, brot):
    """Build and verify a trained factorization at `brot` bits (with the weight refit), log it and repackage."""
    record = {'name': name, 'source': str(source)}
    try:
        folder = WORK / 'builds' / f'{name}_b{brot}' / 'dfthcsos'
        m = RX.lab.build_folder(source, folder, brot=brot, refit=True, r=REF)
        record[f'b{brot}_local'] = {k: m[k] for k in ('error_1', 'error_2', 'Lambda', 'E_gap', 'lambda_eff')}
        summary = RX.verify(folder)
        record.update(brot=brot, folder=str(folder), status=summary.get('status'),
                      cost=summary.get('score'), error=summary.get('error'))
        if summary.get('status') == 'PASS':
            record['improvement'] = RX.BASELINE_COST / summary['score']
    except Exception:
        traceback.print_exc()
        record['status'] = 'ERROR'
    log(record)
    package()


def run_qat(name, source, brot, epochs=20_000):
    """Freeze the vectors at their brot-bit values, retrain only the weights, build, verify and repackage."""
    try:
        _qat(name, source, RX=RX, TQ=TQ, TS=TS, REF=REF, WORK=WORK, brot=brot,
             epochs=max(20, int(epochs * EPOCH_SCALE)))
    except Exception:
        traceback.print_exc()
    package()


def train(name, seed=None, lr_final=1e-6, **cfg):
    """Train, build and verify one config; the learning rate decays exponentially from lr to lr_final."""
    cfg['epochs'] = max(20, int(cfg['epochs'] * EPOCH_SCALE))
    RX.SEED = Path(seed) if seed and Path(seed).exists() else SORTED_SEED
    print(f'=== {name}: seed {RX.SEED.name}, {cfg}')
    TQ.Train = functools.partial(TS.Train, learning_rate_final=lr_final)
    try:
        RX.run(name, cfg, REF)
    except Exception:
        traceback.print_exc()
    package()

## Starting points

In [ ]:
# Eigendecomposition-based starts for (8, 143, C), built from the FCIDUMP (about a minute each)
import df_init

BASE_INIT = seed_npz('init_8_143_44', search_root=TOOLKIT_SRC)   # supplies H1_I and Hsym_lower


def make_init(rank, bases, copies):
    """Write seeds/init_<R>_<B>_<C>.npz with df_init.initialize and return its path."""
    path = WORK / 'seeds' / f'init_{rank}_{bases}_{copies}.npz'
    if not path.exists():
        base = np.load(BASE_INIT)
        _, _, _, S, _ = RX.lab.inputs(BASE_INIT)
        U, W = df_init.initialize(REF['tei'], S, rank, bases, copies)
        np.savez(path, U=U, W=W, H2_I=W.sum(axis=1), H1_I=base['H1_I'], Hsym_lower=base['Hsym_lower'])
    print('init:', path)
    return path


def two_stages(tag, copies):
    """Stage 1 and stage 2 for (8, 143, copies); returns the stage 2 factorization path."""
    seed = make_init(8, 143, copies)
    train(f'{tag}_stage1', seed=seed, rank=8, bases=143, copies=copies, epochs=100_000,
          lr=1e-3, lr_final=1e-5, rho=1e-4, epsilon_dfthc=0.012)
    s1 = WORK / 'runs' / f'{tag}_stage1.npz'
    train(tag, seed=s1 if s1.exists() else seed, rank=8, bases=143, copies=copies, epochs=200_000,
          lr=1e-4, lr_final=1e-6, rho=1e-4, epsilon_dfthc=0.012)
    return WORK / 'runs' / f'{tag}.npz'


def qat_if_needed(tag, source, brot=12):
    """Weight refit at `brot` bits, only when the plain build did not pass."""
    rows = [json.loads(line) for line in open(WORK / 'results.jsonl')] if (WORK / 'results.jsonl').exists() else []
    if source.exists() and not any(r['name'] == tag and r.get('status') == 'PASS' for r in rows):
        run_qat(f'{tag}_q{brot}', source, brot=brot)


## C = 100: stage 1 + stage 2 + refit at 12 bits

In [ ]:
S100 = two_stages('S100', copies=100)
qat_if_needed('S100', S100)

## C = 80: stage 1 + stage 2 + refit at 12 bits

In [ ]:
S80 = two_stages('S80', copies=80)
qat_if_needed('S80', S80)

## Clean up
Keeps `submission.zip`, `results.jsonl`, the trained factorizations (`runs/`) and the builds.

In [ ]:
os.chdir(WORK)
for path in (WORK / 'toolkit', WORK / 'kit', WORK / 'femoco76_ref_cache.npz', WORK / 'seeds'):
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
print(sorted(p.name for p in WORK.iterdir()))